In [2]:
#import packages to run GEE python API

import ee
import geemap
import pprint
import ipyleaflet
import time

# Trigger the authentication flow.
ee.Authenticate()

# Initialize the library
ee.Initialize(project='ee-diptibaral21')

ModuleNotFoundError: No module named 'ee'

In [ ]:
#load packages
import os
import numpy as np
import pandas as pd

In [ ]:
# STEP 0: Setup and Counties

In [ ]:


counties = ee.FeatureCollection("TIGER/2018/Counties")
selected_counties = ['Butte','Colusa','Glenn','Placer','Sacramento','San Joaquin','Sutter','Yolo','Yuba']

sac_valley = counties.filter(ee.Filter.eq('STATEFP', '06')) \
                     .filter(ee.Filter.inList('NAME', selected_counties))


In [ ]:
#visualizing Sac Valley Counties

Map = geemap.Map()
Map.centerObject(sac_valley, 8)  

# Add the layer
Map.addLayer(sac_valley, {},'Sacramento Valley Counties')
Map

Map(center=[39.00285080659933, -121.59337946044376], controls=(WidgetControl(options=['position', 'transparent…

In [ ]:
# STEP 1: Rice Masking

In [ ]:


def mask_rice(image):
    rice = image.select('cropland').eq(3)
    return image.updateMask(rice) \
    .clip(sac_valley) \
    .unmask(0) \
    .set('system:time_start', image.get('system:time_start'))

def get_rice_frequency(start_year, end_year):
    cdl = ee.ImageCollection("USDA/NASS/CDL") \
            .filterDate(f'{start_year}-01-01', f'{end_year}-12-31') \
            .map(lambda img: img.select(['cropland'])) \
            .map(mask_rice)

    rice_sum = cdl.reduce(ee.Reducer.sum())
    rice_frequency = rice_sum.gt(0)
    return rice_sum, rice_frequency
    
rice_sum, rice_frequency = get_rice_frequency(1997, 2023)   
Map.addLayer(rice_sum.clip(sac_valley),{'min':0, 'max': 27, 'palette': ['#e5f5e0', '#a1d99b', '#31a354']},'Rice Frequency (Years)')
Map



Map(bottom=25346.0, center=[39.00285080659933, -121.59337946044376], controls=(WidgetControl(options=['positio…

In [ ]:
# STEP 2: Processing loca Data - filtering from 2030-2100, converting k to C, keeping only tmmx, tmmn, tmean -- script (loca_county_daily_means.py)

In [ ]:
# STEP 3: Calculating GDD - loca_gdd.py

In [ ]:
# STEP 4: Phenology Detection Using Growing Degree Day Model - loca_growth_stages.ipynb 

In [ ]:
# STEP 5: Stress Calculation


# GEE-equivalent county/year/stage stress indices from LOCA2 NetCDFs
# - Reads 20 NetCDF files (10 models × 2 SSPs)
# - Vars: tmmn, tmmx, tmean (dims: time, lat, lon)
# - For each county × year × stage window:
#     * daily stress per pixel (booting/flowering logic)
#     * stage-window climate means (tmmn/tmmx/tmean)
#     * stage-window stress sums (cdstress_bo/cdstress_fl/htstress_fl)
#     * county reduction = AREA-WEIGHTED mean using cos(lat) weights (like equal-area intent in GEE)
# - Exports one CSV per NetCDF file


In [ ]:
#import packages
import os
import re
import glob
import numpy as np
import pandas as pd
import xarray as xr
import geopandas as gpd
import rioxarray
import regionmask

In [ ]:

# thresholds

booting_cold = 13
flowering_cold = 10.9
flowering_heat = 37.5


def calculate_daily_stress(ds: xr.Dataset, stage: str) -> xr.Dataset:
    """
    Compute daily stress for all pixels and all times, but only for the given stage.
    Outside the stage will be handled later by masking dates.
    """
    tmmn = ds["tasmin"]
    tmmx = ds["tasmax"]
    tmean = ds["tmean"]


    # ALWAYS define these so they're never missing
    cdstress_bo = xr.zeros_like(tmmn).rename("cdstress_bo")
    cdstress_fl = xr.zeros_like(tmmn).rename("cdstress_fl")
    htstress_fl = xr.zeros_like(tmmx).rename("htstress_fl")

    if stage == "booting":
        cdstress_bo = xr.where(
            tmmn < booting_cold,
            (booting_cold - tmmn),
            0.0
        ).rename("cdstress_bo")

    elif stage == "flowering":
        cdstress_fl = xr.where(
            tmmn < flowering_cold,
            (flowering_cold - tmmn),
            0.0
        ).rename("cdstress_fl")

        htstress_fl = xr.where(
            tmmx > flowering_heat,
            (tmmx - flowering_heat),
            0.0
        ).rename("htstress_fl")

    elif stage == "grain_fill":
        # keep zeros 
        pass

    else:
        raise ValueError(f"Unknown stage: {stage}")

    return xr.Dataset(
        {
            "tmmn": tmmn,
            "tmmx": tmmx,
            "tmean": tmean,
            "cdstress_bo": cdstress_bo,
            "cdstress_fl": cdstress_fl,
            "htstress_fl": htstress_fl,   
        }
    )


In [ ]:
#County masks + area weights (lat/lon)

def build_county_mask(counties_gdf: gpd.GeoDataFrame, name_col: str = "NAME"):
    """
    Build regionmask Regions from a counties GeoDataFrame.
    """
    gdf = counties_gdf.to_crs("EPSG:4326").copy()

    regions = regionmask.Regions(
        outlines=list(gdf.geometry),
        names=list(gdf[name_col].astype(str).values),
        abbrevs=list(gdf[name_col].astype(str).values),
        name="counties",
    )
    return regions

In [ ]:


def calculate_yearly_stage_stress(stage_feature,
                                  growing_season_images: xr.Dataset,
                                  regions: regionmask.Regions,
                                  mask: xr.DataArray,
                                  weights: xr.DataArray,
                                  name_col: str = "NAME"):
    """
    Python equivalent of EE calculate_yearly_stage_stress(stage_feature).

    Parameters
    ----------
    stage_feature : dict-like or pd.Series
        Must have keys: 'county', 'year', 'stage', 'start_date', 'end_date'
    growing_season_images : xr.Dataset
        Must have vars: 'tmmn','tmmx' (optional 'tmean') and coords 'time'
        Spatial coords should match what mask/weights were built on.
    regions : regionmask.Regions
        Regions created from county polygons (EPSG:4326) using build_county_mask.
    mask : xr.DataArray
        County id mask on the dataset grid (e.g., regions.mask(ds['lon'], ds['lat']))
    weights : xr.DataArray
        Area weights on the grid (e.g., cos(lat) for lat/lon grid), broadcastable to (lat,lon)

    Returns
    -------
    pd.DataFrame
        One-row dataframe with:
        county, stage, year, tmmn, tmmx, tmean, cdstress_bo, cdstress_fl, htstress_fl
        where climate vars are time-means and stress vars are time-sums,
        all spatially averaged over the county polygon.
    """

    # ---- parse stage feature ----
    county = str(stage_feature["county"])
    year   = int(stage_feature["year"])
    stage  = str(stage_feature["stage"])
    start  = pd.to_datetime(stage_feature["start_date"])
    end    = pd.to_datetime(stage_feature["end_date"])

    # ---- county id in the regionmask object ----
    name_to_id = {n: i for i, n in enumerate(regions.names)}
    if county not in name_to_id:
        raise ValueError(f"County '{county}' not found in regions.names.")
    rid = name_to_id[county]

    # ---- subset dataset by stage window ----
    ds_sub = growing_season_images.sel(time=slice(start, end))

    # ---- daily stress (your function; should return ds with tmmn,tmmx,tmean and stress bands) ----
    daily_stress = calculate_daily_stress(ds_sub, stage)  

    # ---- time aggregation (matches EE) ----
    climate_means = daily_stress[["tmmn", "tmmx", "tmean"]].mean("time", skipna=True)
    stress_sums   = daily_stress[["cdstress_bo", "cdstress_fl", "htstress_fl"]].sum("time", skipna=True)

    yearly = xr.merge([climate_means, stress_sums])

    # ---- spatial reduction by county (mean over pixels inside county) ----
    out = {}
    for var in yearly.data_vars:
        da = yearly[var].where(mask == rid)
        # weights expected to be compatible with da's spatial dims ( lat/lon)
        out[var] = float(da.weighted(weights).mean(dim=[d for d in da.dims if d != "time"], skipna=True).values)

    # ---- match EE output fields / properties ----
    out_row = {
        "county": county,
        "stage": stage,
        "year": year,
        "tmmn": out.get("tmmn", np.nan),
        "tmmx": out.get("tmmx", np.nan),
        "tmean": out.get("tmean", np.nan),
        "cdstress_bo": out.get("cdstress_bo", np.nan),
        "cdstress_fl": out.get("cdstress_fl", np.nan),
        "htstress_fl": out.get("htstress_fl", np.nan),
    }

    return pd.DataFrame([out_row])


In [ ]:

# paths
STAGE_TABLE_DIR = "/group/moniergrp/dbaral/run_project/intermediate_data/LOCA_stage_table"
OUT_DIR = "/group/moniergrp/dbaral/run_project/intermediate_data/LOCA_stage_stress_tables"
LOCA_NETCDF_DIR = "/group/moniergrp/dbaral/run_project/input_data/loca2"
SHAPEFILE = "/group/moniergrp/dbaral/run_project/input_data/shape_files/CA_9counties_shapefile.shp"

os.makedirs(OUT_DIR, exist_ok=True)


# counties -- regions
counties_gdf = gpd.read_file(SHAPEFILE).to_crs("EPSG:4326")
regions = build_county_mask(counties_gdf)


# Helper: load matching LOCA dataset


def load_loca_dataset(stage_table_file, netcdf_dir):
    """
    Matches stage table:
      ACCESS-CM2_ssp245_r1i1p1f1__stage_table.csv
    to NetCDF:
      ACCESS-CM2_ssp245_r1i1p1f1_rice_temp.nc
    """
    base = os.path.basename(stage_table_file)
    model = base.split("_ssp")[0]

    m = re.search(r"(ssp\d{3})", base)
    if not m:
        raise ValueError(f"Could not parse SSP from {base}")
    ssp = m.group(1)

    nc_path = os.path.join(netcdf_dir, f"{model}_{ssp}_r1i1p1f1_rice_temp.nc")
    if not os.path.exists(nc_path):
        raise FileNotFoundError(f"Missing NetCDF:\n{nc_path}")

    print(f"[load_loca_dataset] Using: {nc_path}")
    ds = xr.open_dataset(nc_path)

    # --- Kelvin -> Celsius (ACTUALLY applied) ---
    u = str(ds["tasmin"].attrs.get("units", "")).lower()
    vmin = float(ds["tasmin"].min())
    if ("k" in u) or (vmin > 100):
        ds = ds.copy()
        ds["tasmin"] = ds["tasmin"] - 273.15
        ds["tasmax"] = ds["tasmax"] - 273.15
        ds["tasmin"].attrs["units"] = "degC"
        ds["tasmax"].attrs["units"] = "degC"

    # --- coord rename ---
    rename = {}
    if "longitude" in ds.coords and "lon" not in ds.coords:
        rename["longitude"] = "lon"
    if "latitude" in ds.coords and "lat" not in ds.coords:
        rename["latitude"] = "lat"
    if rename:
        ds = ds.rename(rename)

    return ds

    ds = kelvin_to_celsius(ds)
    
    # standardize coord names for regionmask
    rename = {}
    if "longitude" in ds.coords and "lon" not in ds.coords:
        rename["longitude"] = "lon"
    if "latitude" in ds.coords and "lat" not in ds.coords:
        rename["latitude"] = "lat"
    if rename:
        ds = ds.rename(rename)

    return ds

# MAIN LOOP (20 tables)

stage_tables = sorted(glob.glob(os.path.join(STAGE_TABLE_DIR, "*_stage_table.csv")))

for st_path in stage_tables:
    tag = os.path.basename(st_path).replace("_stage_table.csv", "")
    print(f"\nProcessing: {tag}")

    # load stage table
    stage_table_df = pd.read_csv(st_path)
    stage_table_df["start_date"] = pd.to_datetime(stage_table_df["start_date"])
    stage_table_df["end_date"]   = pd.to_datetime(stage_table_df["end_date"])
    stage_table_df["year"]       = stage_table_df["year"].astype(int)

    # load matching LOCA data
    ds = load_loca_dataset(st_path, LOCA_NETCDF_DIR)

    # build mask + weights for THIS grid
    mask = regions.mask(ds["lon"], ds["lat"])          # (lat, lon)
    weights = np.cos(np.deg2rad(ds["lat"]))             # latitude weights

    # ---- EE equivalent: stage_table.map(...).flatten() ----
    rows = []
    for _, r in stage_table_df.iterrows():
        out = calculate_yearly_stage_stress(
            stage_feature=r.to_dict(),
            growing_season_images=ds,
            regions=regions,
            mask=mask,
            weights=weights
        )
        rows.append(out)

    if len(rows) == 0:
        print(f"No stress calculated for {tag}")
        all_year_table = pd.DataFrame(columns=["county", "year", "stage", "cdstress_bo", "cdstress_fl", "htstress_fl"])
    else:
        all_year_table = pd.concat(rows, ignore_index=True)


    # ---- EE equivalent: .select(...) ----
    all_year_table_selected = all_year_table[
        ["county", "year", "stage",
         "cdstress_bo", "cdstress_fl", "htstress_fl"]
    ]

    # ---- EE equivalent: first().getInfo() ----
    first_export_feature = all_year_table_selected.iloc[0].to_dict()
    print("First feature:", first_export_feature)

    # save
    out_csv = os.path.join(OUT_DIR, f"{tag}_yearly_stage_stress.csv")
    all_year_table_selected.to_csv(out_csv, index=False)
    print(f"Saved → {out_csv}")




Processing: ACCESS-CM2_ssp245_r1i1p1f1_
[load_loca_dataset] Using: /group/moniergrp/dbaral/run_project/input_data/loca2/ACCESS-CM2_ssp245_r1i1p1f1_rice_temp.nc
First feature: {'county': 'Butte', 'year': 2030, 'stage': 'booting', 'cdstress_bo': 0.0, 'cdstress_fl': 0.0, 'htstress_fl': 0.0}
Saved → /group/moniergrp/dbaral/run_project/intermediate_data/LOCA_stage_stress_tables/ACCESS-CM2_ssp245_r1i1p1f1__yearly_stage_stress.csv

Processing: ACCESS-CM2_ssp585_r1i1p1f1_
[load_loca_dataset] Using: /group/moniergrp/dbaral/run_project/input_data/loca2/ACCESS-CM2_ssp585_r1i1p1f1_rice_temp.nc
First feature: {'county': 'Butte', 'year': 2030, 'stage': 'booting', 'cdstress_bo': 0.0, 'cdstress_fl': 0.0, 'htstress_fl': 0.0}
Saved → /group/moniergrp/dbaral/run_project/intermediate_data/LOCA_stage_stress_tables/ACCESS-CM2_ssp585_r1i1p1f1__yearly_stage_stress.csv

Processing: CNRM-ESM2-1_ssp245_r1i1p1f1_
[load_loca_dataset] Using: /group/moniergrp/dbaral/run_project/input_data/loca2/CNRM-ESM2-1_ssp245_r